In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/tiempoDS.csv.gz')
df

FileNotFoundError: [Errno 2] No such file or directory: '../data/tiempoDS.csv.gz'

In [12]:
# import pandas as pd
# import numpy as np
# from pmdarima.arima import auto_arima
# from joblib import Parallel, delayed
# import warnings
# warnings.filterwarnings("ignore")

# def train_country_robust(country_name, df_country):
#     try:
#         # 1. Preparación y Filtrado Temporal
#         df = df_country.copy()
        
#         # Crear índice datetime
#         df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))
#         df.set_index('date', inplace=True)
#         df.sort_index(inplace=True)
        
#         # --- FILTRADO CLAVE: Solo desde 1950 ---
#         df = df[df.index >= '1950-01-01']
        
#         if len(df) < 24: # Mínimo 2 años de datos post-1950
#             return None

#         # Eliminar duplicados de fecha (seguridad)
#         df = df[~df.index.duplicated(keep='first')]
        
#         # Rellenar huecos temporales si faltan meses entre 1950 y ahora
#         full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq='MS')
#         df = df.reindex(full_range)
#         df.index.name = 'date'
        
#         # Interpolar valores faltantes
#         df['AvTemp'] = df['AvTemp'].interpolate(method='linear', limit_area='inside')
#         df.dropna(subset=['AvTemp'], inplace=True)
        
#         if len(df) < 24:
#             return None

#         # 2. Entrenamiento SARIMAX
#         model = auto_arima(
#             df['AvTemp'],
#             m=12, seasonal=True,
#             d=None, D=None, 
#             max_p=3, max_q=3,
#             max_P=2, max_Q=2,
#             stepwise=True,
#             suppress_warnings=True,
#             error_action='ignore',
#             trace=False
#         )
        
#         # 3. Predicción (30 años)
#         n_periods = 30 * 12
#         forecast, conf_int = model.predict(n_periods=n_periods, return_conf_int=True)
        
#         last_date = df.index[-1]
#         future_dates = pd.date_range(start=last_date + pd.offsets.MonthBegin(1), periods=n_periods, freq='MS')
        
#         df_res = pd.DataFrame({
#             'Country': country_name,
#             'Date': future_dates,
#             'Year': future_dates.year,
#             'Predicted_Temp': forecast,
#             'Conf_Int_Lower': conf_int[:, 0],
#             'Conf_Int_Upper': conf_int[:, 1],
#             'AIC': model.aic()
#         })
        
#         return df_res

#     except Exception as e:
#         return None

# # --- EJECUCIÓN ---
# print("Filtrando datos desde 1950 y procesando...")

# # Asegúrate de que tu DataFrame original se llama 'df'
# grouped = df.groupby('country') 

# results = Parallel(n_jobs=-1, verbose=10)(
#     delayed(train_country_robust)(name, group) 
#     for name, group in grouped
# )

# valid_results = [r for r in results if r is not None]

# if valid_results:
#     final_df = pd.concat(valid_results, ignore_index=True)
#     final_df.to_csv('predicciones_climaticas_1950_plus.csv', index=False)
#     print(f"\n✅ Éxito: {len(valid_results)} países procesados con datos post-1950.")
# else:
#     print("❌ No se generaron predicciones.")

In [13]:
# import pandas as pd
# import numpy as np
# from pmdarima.arima import auto_arima
# from statsmodels.tsa.stattools import adfuller
# from statsmodels.tsa.seasonal import seasonal_decompose
# from joblib import Parallel, delayed
# from tqdm import tqdm # Importamos tqdm para la barra de progreso
# import warnings
# import time

# from tqdm.autonotebook import tqdm as tqdm_notebook
# warnings.filterwarnings("ignore")

In [14]:
# def analyze_and_predict_country(country_name, df_country):
#     try:
#         # 1. Preparación de Datos
#         df = df_country.copy()
#         df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))
#         df.set_index('date', inplace=True)
#         df.sort_index(inplace=True)
        
#         # Filtrado Post-1950
#         df = df[df.index >= '1950-01-01']
        
#         if len(df) < 60: 
#             return None

#         # Seguridad mínima para índice único
#         df = df[~df.index.duplicated(keep='first')]
        
#         # Interpolación mínima para continuidad mensual (necesaria para SARIMA/Decompose)
#         full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq='MS')
#         df = df.reindex(full_range)
#         df['AvTemp'] = df['AvTemp'].interpolate(method='linear', limit_area='inside')
#         df.dropna(subset=['AvTemp'], inplace=True)

#         if len(df) < 60:
#             return None

#         # 2. Separación Train (1950-1999) / Test (2000-2012)
#         train_data = df[df.index < '2000-01-01']['AvTemp']
#         test_data = df[df.index >= '2000-01-01']['AvTemp']
        
#         if len(train_data) < 24 or len(test_data) < 12:
#             return None

#         # 3. Análisis Estadístico Previo (Sobre Train)
        
#         # A. Estacionariedad (ADF Test)
#         adf_result = adfuller(train_data.dropna())
#         adf_pvalue = adf_result[1]
#         is_stationary = adf_pvalue < 0.05
        
#         # B. Descomposición Estacional
#         try:
#             decomposition = seasonal_decompose(train_data, model='additive', period=12)
#             trend_mean = decomposition.trend.mean()
#             seasonal_strength = decomposition.seasonal.std()
#         except:
#             trend_mean = np.nan
#             seasonal_strength = np.nan

#         # 4. Modelado SARIMA (Entrenamiento)
#         model = auto_arima(
#             train_data,
#             m=12, seasonal=True,
#             d=None, D=None, 
#             max_p=3, max_q=3,
#             max_P=2, max_Q=2,
#             stepwise=True,
#             suppress_warnings=True,
#             error_action='ignore',
#             trace=False
#         )
        
#         order = model.order
#         seasonal_order = model.seasonal_order
#         aic = model.aic()

#         # 5. Validación en Test (2000-2012)
#         forecast_test = model.predict(n_periods=len(test_data))
#         rmse = np.sqrt(np.mean((test_data.values - forecast_test)**2))
#         mae = np.mean(np.abs(test_data.values - forecast_test))
        
#         # --- DEVOLVER RESULTADOS DE VALIDACIÓN (Sin predicción futura) ---
#         df_stats = pd.DataFrame({
#             'Country': [country_name], # Importante: lista para crear fila
#             'RMSE_Test': [rmse],
#             'MAE_Test': [mae],
#             'AIC': [aic],
#             'Model_Order': [str(order)],
#             'Seasonal_Order': [str(seasonal_order)],
#             'ADF_PValue': [adf_pvalue],
#             'Is_Stationary': [is_stationary],
#             'Trend_Mean_Train': [trend_mean],
#             'Seasonal_Strength': [seasonal_strength]
#         })
        
#         return df_stats
    
#         # # 6. Predicción Futura (Reentrenamiento con TODOS los datos 1950-2012)
#         # model_final = auto_arima(
#         #     df['AvTemp'],
#         #     m=12, seasonal=True,
#         #     order=order, seasonal_order=seasonal_order,
#         #     stepwise=False,
#         #     suppress_warnings=True,
#         #     error_action='ignore'
#         # )
        
#         # n_periods_future = 30 * 12
#         # forecast_future, conf_int = model_final.predict(n_periods=n_periods_future, return_conf_int=True)
        
#         # last_date = df.index[-1]
#         # future_dates = pd.date_range(start=last_date + pd.offsets.MonthBegin(1), periods=n_periods_future, freq='MS')
        
#         # # 7. Construcción del Resultado
#         # df_res = pd.DataFrame({
#         #     'Country': country_name,
#         #     'Date': future_dates,
#         #     'Year': future_dates.year,
#         #     'Predicted_Temp': forecast_future,
#         #     'Conf_Int_Lower': conf_int[:, 0],
#         #     'Conf_Int_Upper': conf_int[:, 1],
#         #     'RMSE_Test': rmse,
#         #     'MAE_Test': mae,
#         #     'AIC': aic,
#         #     'Model_Order': str(order),
#         #     'Seasonal_Order': str(seasonal_order),
#         #     'ADF_PValue': adf_pvalue,
#         #     'Is_Stationary': is_stationary,
#         #     'Trend_Mean_Train': trend_mean,
#         #     'Seasonal_Strength': seasonal_strength
#         # })
        
#         # return df_res

#     except Exception as e:
#         print(f"Error en {country_name}: {e}")
#         return None
    
# # --- EJECUCIÓN CON BARRA DE PROGRESO ---
# print("Iniciando análisis estadístico y validación (Fase 1)...")

# grouped = df.groupby('country') 
# countries_list = list(grouped.groups.keys())
# total_countries = len(countries_list)

# # Ejecución paralela monitorizada
# results = Parallel(n_jobs=-1, backend='loky')(
#     delayed(analyze_and_predict_country)(name, group) 
#     for name, group in tqdm(grouped, total=total_countries, desc="Clima ML")
# )

# valid_results = [r for r in results if r is not None]

# if valid_results:
#     # Concatenar los DataFrames de estadísticas
#     stats_df = pd.concat(valid_results, ignore_index=True)
    
#     # Guardar resumen de calidad del modelo
#     stats_df.to_csv('resumen_validacion_modelos.csv', index=False)
    
#     print(f"\n✅ Éxito: {len(valid_results)} países analizados.")
#     print("\n--- Resumen de Validación (Test 2000-2012) ---")
#     print(f"RMSE Promedio: {stats_df['RMSE_Test'].mean():.4f}")
#     print(f"MAE Promedio: {stats_df['MAE_Test'].mean():.4f}")
    
#     # Mostrar los 5 peores modelos
#     print("\n⚠️ Top 5 países con mayor error (RMSE):")
#     print(stats_df[['Country', 'RMSE_Test', 'ADF_PValue']].sort_values(by='RMSE_Test', ascending=False).head())
    
# else:
#     print("❌ No se generaron resultados válidos.")

RMSE Promedio (2.62°C): Es un valor aceptable para temperaturas mensuales a nivel global, considerando la variabilidad natural del clima. Significa que, de media, el modelo se equivoca en ±2.6 grados al predecir meses que "ya han ocurrido" (2000-2012) usando solo datos antiguos.
Los "Outliers" (Top 5 Errores):

   * Rusia, China, Kazajistán, Irán: Son países masivos con climas continentales extremos. Un solo promedio nacional (AvTemp) para toda Rusia es estadísticamente muy difícil de modelar porque mezcla el Ártico con el subtópico. El alto RMSE aquí no significa necesariamente que el modelo sea malo, sino que la serie temporal es muy "ruidosa" o heterogénea.
   * Jamaica: Al ser tropical, tiene menos estacionalidad térmica pero mucha influencia de fenómenos como huracanes o El Niño, que son difíciles de capturar solo con SARIMA.

ADF P-Value: Todos son extremadamente bajos (< 0.05). Esto confirma que las series son estacionarias después de las diferencias que aplicó auto_arima (o que la tendencia es muy fuerte y el modelo la está gestionando bien).

In [15]:
import pandas as pd
import numpy as np
from pmdarima.arima import auto_arima
from statsmodels.tsa.seasonal import seasonal_decompose
from joblib import Parallel, delayed, dump, load
from tqdm import tqdm
import warnings
import os
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# Crear carpeta para guardar los modelos
MODEL_DIR = "../modelos/modelos_climaticos_sarima"
if not os.path.exists(MODEL_DIR):
    os.makedirs(MODEL_DIR)

In [ ]:
def train_and_save_robust(country_name, df_country):
    try:
        # 1. Preparación de Datos
        df = df_country.copy()
        df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))
        df.set_index('date', inplace=True)
        df.sort_index(inplace=True)
        
        # Filtrado Post-1950
        df = df[df.index >= '1950-01-01']
        if len(df) < 60: return None
        
        # Limpieza mínima
        df = df[~df.index.duplicated(keep='first')]
        full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq='MS')
        df = df.reindex(full_range)
        df['AvTemp'] = df['AvTemp'].interpolate(method='linear', limit_area='inside')
        df.dropna(subset=['AvTemp'], inplace=True)
        if len(df) < 60: return None

        # 2. Descomposición Estacional para aislar Tendencia
        # Usamos 'additive' porque la amplitud estacional parece constante en la mayoría de países
        decomp = seasonal_decompose(df['AvTemp'], model='additive', period=12, extrapolate_trend='freq')
        
        # La serie "limpia" para entrenar es: Observado - Tendencia
        # Así el modelo solo se preocupa de la estacionalidad y el ruido
        series_to_train = df['AvTemp'] - decomp.trend
        
        # 3. Split Train/Test
        train_data = series_to_train[series_to_train.index < '2000-01-01']
        test_data_clean = series_to_train[series_to_train.index >= '2000-01-01']
        test_real_temp = df['AvTemp'][df.index >= '2000-01-01'] # Para calcular error real
        
        if len(train_data) < 24 or len(test_data_clean) < 12: return None

        # 4. Entrenamiento SARIMA sobre la serie destendenciada
        model = auto_arima(
            train_data.dropna(), 
            m=12, seasonal=True,
            start_p=1, start_q=1,
            max_p=3, max_q=3,
            start_P=1, start_Q=1, # Forzamos componentes estacionales
            max_P=2, max_Q=2,
            d=None, D=None,
            stepwise=True, 
            suppress_warnings=True, 
            error_action='ignore'
        )
        
        # 5. Validación en Test
        # Predicimos la parte "limpia" y le sumamos la tendencia real del periodo test
        forecast_clean = model.predict(n_periods=len(test_data_clean))
        trend_test = decomp.trend[decomp.trend.index >= '2000-01-01']
        
        # Reconstrucción de la predicción final
        forecast_final = forecast_clean + trend_test.values
        
        residuals = test_real_temp.values - forecast_final
        rmse = np.sqrt(np.mean(residuals**2))
        
        # 6. Guardar el Modelo y la Descomposición (para poder reconstruir futuro)
        safe_name = country_name.replace(" ", "_").replace("/", "_")
        model_path = os.path.join(MODEL_DIR, f"{safe_name}.joblib")
        
        # Guardamos un diccionario con todo lo necesario para predecir el futuro
        artifact = {
            'model': model,
            'last_trend_value': decomp.trend.iloc[-1], # Último valor de tendencia conocido
            'trend_slope': (decomp.trend.iloc[-1] - decomp.trend.iloc[-12]) / 12, # Pendiente anual aprox
            'order': model.order,
            'seasonal_order': model.seasonal_order,
            'rmse': rmse
        }
        dump(artifact, model_path)
        
        # Devolvemos métricas para el DataFrame de diagnóstico
        return {
            'Country': country_name,
            'RMSE': rmse,
            'Model_Path': model_path,
            'Order': str(model.order),
            'Seasonal_Order': str(model.seasonal_order)
        }

    except Exception as e:
        print(f"Error en {country_name}: {e}")
        return None

In [17]:
print("Iniciando entrenamiento robusto con destendencia y guardado...")

grouped = df.groupby('country')
total_countries = len(list(grouped.groups.keys()))

results = Parallel(n_jobs=-1, backend='loky')(
    delayed(train_and_save_robust)(name, group) 
    for name, group in tqdm(grouped, total=total_countries, desc="Entrenando Países")
)

valid_results = [r for r in results if r is not None]

if valid_results:
    metrics_df = pd.DataFrame(valid_results)
    metrics_df.to_csv('metricas_modelos_guardados.csv', index=False)
    
    print(f"\n✅ Éxito: {len(valid_results)} modelos entrenados y guardados en '{MODEL_DIR}'")
    
    # Mostrar mejores y peores
    print("\n--- Top 5 MEJORES Modelos (Menor RMSE) ---")
    print(metrics_df.sort_values(by='RMSE').head())
    
    print("\n--- Top 5 PEORES Modelos (Mayor RMSE) ---")
    print(metrics_df.sort_values(by='RMSE', ascending=False).head())
else:
    print("❌ No se generaron modelos.")

Iniciando entrenamiento robusto con destendencia y guardado...


Entrenando Países: 100%|██████████| 158/158 [35:18<00:00, 13.41s/it]



✅ Éxito: 123 modelos entrenados y guardados en '../modelos/modelos_climaticos_sarima'

--- Top 5 MEJORES Modelos (Menor RMSE) ---
         Country      RMSE                                         Model_Path  \
29    Costa Rica  0.284312  ../modelos/modelos_climaticos_sarima\Costa_Ric...   
55        Guyana  0.297651  ../modelos/modelos_climaticos_sarima\Guyana.jo...   
109  Puerto Rico  0.324123  ../modelos/modelos_climaticos_sarima\Puerto_Ri...   
86     Mauritius  0.329616  ../modelos/modelos_climaticos_sarima\Mauritius...   
77       Liberia  0.347488  ../modelos/modelos_climaticos_sarima\Liberia.j...   

         Order Seasonal_Order  
29   (0, 0, 1)  (1, 0, 1, 12)  
55   (1, 0, 2)  (2, 0, 2, 12)  
109  (3, 0, 1)  (2, 0, 1, 12)  
86   (3, 0, 1)  (2, 0, 2, 12)  
77   (3, 0, 1)  (1, 0, 1, 12)  

--- Top 5 PEORES Modelos (Mayor RMSE) ---
         Country       RMSE  \
71    Kazakhstan  11.776230   
112       Russia   9.455876   
0    Afghanistan   8.811317   
63          Iran   8.45

In [30]:
import matplotlib.pyplot as plt
import pandas as pd
import joblib
import os
from statsmodels.tsa.seasonal import seasonal_decompose

def plot_diagnosis_correct(country_name):
    # 1. Cargar histórico ordenado
    hist = df[df['country'] == country_name].copy()
    hist['date'] = pd.to_datetime(hist[['year', 'month']].assign(day=1))
    hist = hist.sort_values('date') # CLAVE: Ordenar cronológicamente
    
    # Filtramos solo el periodo de test (2000-2012) para comparar con la validación
    hist_test = hist[hist['date'] >= '2000-01-01']
    
    # 2. Cargar modelo guardado
    safe_name = country_name.replace(" ", "_").replace("/", "_")
    model_path = os.path.join(MODEL_DIR, f"{safe_name}.joblib")
    
    if not os.path.exists(model_path):
        print(f"No se encontró modelo para {country_name}")
        return
        
    artifact = joblib.load(model_path)
    model = artifact['model']
    
    # 3. Reconstruir predicción para el periodo test
    # Necesitamos la tendencia del periodo test para sumararla a la predicción limpia
    hist_series = hist_test.set_index('date')['AvTemp']
    
    # Descomponemos el histórico test para obtener su tendencia específica
    decomp_hist = seasonal_decompose(hist_series, model='additive', period=12, extrapolate_trend='freq')
    
    # Serie limpia (sin tendencia) para predecir
    series_clean_test = hist_series - decomp_hist.trend
    
    # Predicción del modelo sobre la serie limpia
    forecast_clean = model.predict(n_periods=len(series_clean_test))
    
    # Reconstrucción final: Predicción limpia + Tendencia histórica
    forecast_final = forecast_clean + decomp_hist.trend.values
    
    # 4. Graficar CORRECTAMENTE
    plt.figure(figsize=(14, 6))
    
    # Línea Real (Azul continua)
    plt.plot(hist_test['date'], hist_test['AvTemp'], 
             label=f'Temperatura Real ({country_name})', 
             color='#1f77b4', linewidth=1.5, alpha=0.8)
    
    # Línea Predicción (Roja punteada)
    plt.plot(hist_test['date'], forecast_final, 
             label=f'Predicción Modelo (RMSE: {artifact["rmse"]:.2f})', 
             color='#d62728', linestyle='--', linewidth=1.5)
    
    plt.title(f"Validación Temporal Continua: {country_name} (2000-2012)", fontsize=14)
    plt.xlabel("Año", fontsize=12)
    plt.ylabel("Temperatura Media (°C)", fontsize=12)
    plt.legend(loc='upper left')
    plt.grid(True, linestyle=':', alpha=0.6)
    
    # Formato de fecha para que se vea limpio
    plt.gcf().autofmt_xdate()
    
    plt.tight_layout()
    plt.show()

# Prueba con Somalia (el "mejor ajuste" que se veía mal)
plot_diagnosis_correct('Spain')

# Prueba con Kazajistán (el "peor ajuste")
plot_diagnosis_correct('Kazakhstan')

OutOfBoundsDatetime: Out of bounds timestamp: 2649-12-01 00:00:00 with frequency 'ns'